# Cleaning Experiments

This notebook creates **2 custom datasets** for experiment purposes:

| # | Dataset | Methods applied |
|---|---|---|
| 1 | `dataset_experiment_01_all_except_urls.csv` | All methods **except** `remove_urls` |
| 2 | `dataset_experiment_02_subset.csv` | Only: `remove_mentions`, `remove_non_text_comments`, `remove_duplicate_comments`, `remove_outliers` |

**Input:** Original CSV file from Google Drive.

**Output:** 2 CSV files saved in the same Drive folder as the original dataset.

In [ ]:
# Block 1: Clone repo and install dependencies
!git clone -b experiment https://github.com/TrieuKhac-dev/Toxic-Comment-Classification.git
%cd Toxic-Comment-Classification

!pip install -q underthesea wordcloud matplotlib seaborn pandas emoji stopwordsiso nltk scikit-learn gdown requests

In [ ]:
# Block 2: Imports
import sys
import os
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import shutil

warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath("."))

from src.dataset.loader import download_from_drive, read_csv_with_columns
from src.dataset.cleaning import (
    remove_html_and_entities,
    remove_urls,
    remove_mentions,
    remove_emoji,
    remove_special_chars,
    remove_null_or_empty,
    remove_non_text_comments,
    remove_duplicate_comments,
    remove_outliers,
)
from src.dataset.feature_enginering import (
    add_length_features,
    add_punctuation_emoji_features,
)
from src.dataset.preprocessing import normalize_text
from config.dataset_config import default_dataset_config
from config.cleaning_config import default_cleaning_config

print("Import successful!")

In [ ]:
# Block 3: Mount Google Drive and copy dataset locally
from google.colab import drive
drive.mount("/content/drive")

# --- EDIT THESE VALUES ---
DRIVE_SOURCE = "/content/drive/MyDrive/CommentClassificationDataset/raw_dataset.csv"
COMMENT_COL = "comment"       # Name of comment column
LABEL_COL = "is_toxic"        # Name of label column
# --------------------------

# Copy file from Drive to local
LOCAL_PATH = "dataset/raw/raw_dataset.csv"
os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)
shutil.copy2(DRIVE_SOURCE, LOCAL_PATH)
print(f"Copied file from Drive to {LOCAL_PATH}")

# Output folder on Drive (same folder as source)
DRIVE_OUTPUT_DIR = os.path.dirname(DRIVE_SOURCE)
print(f"Output Drive folder: {DRIVE_OUTPUT_DIR}")

In [ ]:
# Block 4: Read original dataset
df_original = read_csv_with_columns(
    data_path=LOCAL_PATH,
    comment_col=COMMENT_COL,
    label_col=LABEL_COL,
)

print(f"\nOriginal dataset shape: {df_original.shape}")
print(f"Columns: {df_original.columns.tolist()}")

---
## Experiment 1: All methods EXCEPT remove_urls

Applies the full cleaning pipeline from `dataset_cleaned_10_all_methods_combined` but **skips** `remove_urls`.

Pipeline order:
1. `remove_html_and_entities`
2. ~~`remove_urls`~~ (skipped)
3. `remove_mentions`
4. `remove_emoji`
5. `remove_special_chars`
6. `remove_null_or_empty`
7. `remove_non_text_comments`
8. `remove_duplicate_comments`
9. `remove_outliers`

In [ ]:
# ============================================================
# Experiment 1: All methods EXCEPT remove_urls
# ============================================================
print("=" * 60)
print("Experiment 1: All methods EXCEPT remove_urls")
print("=" * 60)

df = df_original.copy()
initial_rows = len(df)

# ---- Step 1: Text-level transformations ----
print("\n[Step 1/4] remove_html_and_entities...")
df[COMMENT_COL] = df[COMMENT_COL].astype(str).apply(remove_html_and_entities)

print("[Step 1/4] remove_urls... SKIPPED")

print("[Step 1/4] remove_mentions...")
df[COMMENT_COL] = df[COMMENT_COL].apply(remove_mentions)

print("[Step 1/4] remove_emoji...")
df[COMMENT_COL] = df[COMMENT_COL].apply(remove_emoji)

print("[Step 1/4] remove_special_chars...")
df[COMMENT_COL] = df[COMMENT_COL].apply(
    lambda x: remove_special_chars(x, keep_punctuation=default_cleaning_config.keep_punctuation)
)

# ---- Step 2: Remove null/empty rows ----
print("\n[Step 2/4] remove_null_or_empty...")
df, report_null = remove_null_or_empty(
    df,
    comment_col=COMMENT_COL,
    label_col=LABEL_COL,
    max_null_label_ratio=default_cleaning_config.max_null_label_ratio,
)
print(f"  Removed {report_null['empty_comment_removed']} empty comments")
print(f"  Removed {report_null['null_label_removed']} null label rows")

# ---- Step 3: Remove non-text comments ----
print("\n[Step 3/4] remove_non_text_comments...")
df, report_non_text = remove_non_text_comments(
    df,
    comment_col=COMMENT_COL,
)
print(f"  Removed {report_non_text['non_text_removed']} non-text comments")

# ---- Step 4: Remove duplicates ----
print("\n[Step 4/4] remove_duplicate_comments...")
df, report_dup = remove_duplicate_comments(
    df,
    comment_col=COMMENT_COL,
    label_col=LABEL_COL,
)
print(f"  Duplicate groups: {report_dup['duplicate_comment_groups']}")
print(f"  Removed (different labels): {report_dup['removed_rows_different_labels']}")
print(f"  Removed (same label): {report_dup['removed_rows_duplicate_same_label']}")

# ---- Step 5: Remove outliers ----
print("\n[Step 5/5] remove_outliers (Isolation Forest)...")
df_temp = add_length_features(df, comment_col=COMMENT_COL)
df_temp = add_punctuation_emoji_features(df_temp, comment_col=COMMENT_COL)

df, report_out = remove_outliers(
    df_temp,
    feature_cols=default_cleaning_config.outlier_feature_cols,
    contamination=default_cleaning_config.outlier_contamination,
    random_state=default_cleaning_config.outlier_random_state,
    label_col=LABEL_COL,
)

# Drop temporary feature columns
df = df.drop(
    columns=["word_len", "char_len", "num_exclamation", "num_question", "num_upper", "num_emoji"],
    errors="ignore",
)
print(f"  Removed {report_out['outlier_count']} outliers ({report_out['outlier_ratio']:.2%})")

# ---- Summary ----
print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Initial rows: {initial_rows}")
print(f"Final rows:   {len(df)}")
print(f"Removed:      {initial_rows - len(df)} rows")

output_name = "dataset_experiment_01_all_except_urls.csv"
output_path = os.path.join(DRIVE_OUTPUT_DIR, output_name)
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\nSaved: {output_path}")
print()

---
## Experiment 2: Subset of methods

Applies only 4 specific methods:
1. `remove_mentions` (text-level)
2. `remove_non_text_comments` (dataset-level)
3. `remove_duplicate_comments` (dataset-level)
4. `remove_outliers` (dataset-level)

In [ ]:
# ============================================================
# Experiment 2: Subset of methods
# Only: remove_mentions, remove_non_text_comments,
#        remove_duplicate_comments, remove_outliers
# ============================================================
print("=" * 60)
print("Experiment 2: Subset (mentions + non-text + duplicates + outliers)")
print("=" * 60)

df = df_original.copy()
initial_rows = len(df)

# ---- Step 1: Text-level ----
print("\n[Step 1/4] remove_mentions...")
df[COMMENT_COL] = df[COMMENT_COL].astype(str).apply(remove_mentions)

# ---- Step 2: Remove non-text comments ----
print("\n[Step 2/4] remove_non_text_comments...")
df, report_non_text = remove_non_text_comments(
    df,
    comment_col=COMMENT_COL,
)
print(f"  Removed {report_non_text['non_text_removed']} non-text comments")

# ---- Step 3: Remove duplicates ----
print("\n[Step 3/4] remove_duplicate_comments...")
df, report_dup = remove_duplicate_comments(
    df,
    comment_col=COMMENT_COL,
    label_col=LABEL_COL,
)
print(f"  Duplicate groups: {report_dup['duplicate_comment_groups']}")
print(f"  Removed (different labels): {report_dup['removed_rows_different_labels']}")
print(f"  Removed (same label): {report_dup['removed_rows_duplicate_same_label']}")

# ---- Step 4: Remove outliers ----
print("\n[Step 4/4] remove_outliers (Isolation Forest)...")
df_temp = add_length_features(df, comment_col=COMMENT_COL)
df_temp = add_punctuation_emoji_features(df_temp, comment_col=COMMENT_COL)

df, report_out = remove_outliers(
    df_temp,
    feature_cols=default_cleaning_config.outlier_feature_cols,
    contamination=default_cleaning_config.outlier_contamination,
    random_state=default_cleaning_config.outlier_random_state,
    label_col=LABEL_COL,
)

# Drop temporary feature columns
df = df.drop(
    columns=["word_len", "char_len", "num_exclamation", "num_question", "num_upper", "num_emoji"],
    errors="ignore",
)
print(f"  Removed {report_out['outlier_count']} outliers ({report_out['outlier_ratio']:.2%})")

# ---- Summary ----
print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Initial rows: {initial_rows}")
print(f"Final rows:   {len(df)}")
print(f"Removed:      {initial_rows - len(df)} rows")

output_name = "dataset_experiment_02_subset.csv"
output_path = os.path.join(DRIVE_OUTPUT_DIR, output_name)
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\nSaved: {output_path}")
print()

---
## Summary

Created 2 experiment datasets:

| # | File name | Methods applied |
|---|---|---|
| 1 | `dataset_experiment_01_all_except_urls.csv` | All methods **except** `remove_urls` |
| 2 | `dataset_experiment_02_subset.csv` | `remove_mentions` + `remove_non_text_comments` + `remove_duplicate_comments` + `remove_outliers` |

All saved in the same Drive folder as the original dataset.